In [1]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
from bs4 import BeautifulSoup
import datetime
import os
import re
from time import sleep
from urllib.parse import urljoin

import pandas as pd
import requests

In [2]:
#------------------------------------------------ Begin_fileName ----------------------------------------
regulatorName = 'MW RBM'

print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now = datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":", ".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment
os.chdir(scriptfolder)

tempfolder = os.path.join(scriptfolder, 'tempfolder')
if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

Running MW RBM Web Scraping Tool v.1.1


In [3]:
#------------------------------------------------ Begin_Variable ----------------------------------------
sqldict = {
    'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [],
    'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],
    'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [],
    'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [],
    'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [],
    'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [],
    'RegCtry': [], 'RegCode': [], 'ListCode': [], 'ListLanguage': [],
    'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [],
    'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
    'Address_1 - Mother company': [], 'Address_2 -  Mother company': [],
    'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [],
    'Phone - Mother company': [], 'Check': []
}

processdate = now.strftime('%Y-%m-%d')

regdict = {
    regulatorName + ' 1': 'https://www.rbm.mw/Supervision/BankSupervision/?activeTab=BASURegisteredBanks',
    regulatorName + ' 2': 'https://www.rbm.mw/Supervision/PensionsandInsurance/?activeTab=PISULicensedEntities',
}

Typology = {
    regulatorName + ' 1': 'List of Banks & Other Financial Institutions',
    regulatorName + ' 2': 'Pensions & Insurance Supervision',
}

In [4]:
#------------------------------------------------ Begin_Function ----------------------------------------
EMAIL_RE = re.compile(r'[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}', re.IGNORECASE)


def bourange_same_length_array(sqldict):
    maxlen = len(sqldict['ListProcessDate'])
    for key in sqldict:
        if len(sqldict[key]) != maxlen:
            sqldict[key] = sqldict[key] + [''] * (maxlen - len(sqldict[key]))
    return sqldict


def clean_text(text):
    return re.sub(r'\s+', ' ', str(text or '')).strip()


def normalise_heading(text):
    text = clean_text(text).lower().replace('&', 'and')
    return re.sub(r'[^a-z0-9 ]+', '', text)


def fetch_soup(url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                      '(KHTML, like Gecko) Chrome/124.0 Safari/537.36'
    }
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    return BeautifulSoup(response.text, 'html.parser')


def find_heading(soup, heading_text):
    target = normalise_heading(heading_text)
    for heading in soup.find_all(re.compile(r'^h[1-6]$')):
        if target in normalise_heading(heading.get_text(' ', strip=True)):
            return heading
    return None


def iter_section_nodes(soup, start_heading, stop_headings):
    start = find_heading(soup, start_heading)
    if start is None:
        print(f"[WARN] : Section not found - {start_heading}")
        return

    stops = [normalise_heading(stop) for stop in stop_headings]
    for node in start.find_all_next():
        if not getattr(node, 'name', None):
            continue

        if re.fullmatch(r'h[1-6]', node.name or ''):
            node_heading = normalise_heading(node.get_text(' ', strip=True))
            if any(stop in node_heading for stop in stops):
                break
            yield node
            continue

        if node.name in ('li', 'table', 'p', 'strong', 'b'):
            yield node


def category_from_node(node):
    if node.name in ('h4', 'h5', 'h6'):
        return clean_text(node.get_text(' ', strip=True))

    if node.name == 'p':
        marker = node.find(['strong', 'b'])
        if marker:
            text = clean_text(marker.get_text(' ', strip=True))
            if text and len(text) <= 90:
                return text

    if node.name in ('strong', 'b') and node.parent and node.parent.name != 'li':
        text = clean_text(node.get_text(' ', strip=True))
        if text and len(text) <= 90:
            return text

    return ''


def unique_join(values):
    seen = []
    for value in values:
        value = clean_text(value)
        if value and value not in seen:
            seen.append(value)
    return '; '.join(seen)


def extract_contact_links(node):
    websites = []
    emails = []
    for link in node.find_all('a'):
        href = clean_text(link.get('href', ''))
        text = clean_text(link.get_text(' ', strip=True))
        if href.lower().startswith('mailto:'):
            emails.append(href.split(':', 1)[1])
        elif href.lower().startswith(('http://', 'https://')):
            websites.append(href)
        elif EMAIL_RE.search(text):
            emails.extend(EMAIL_RE.findall(text))

    plain_text = node.get_text(' ', strip=True)
    emails.extend(EMAIL_RE.findall(plain_text))
    return unique_join(websites), unique_join(emails)


def clean_entity_name(node):
    text = clean_text(node.get_text(' ', strip=True))
    candidate_link_names = []

    for link in node.find_all('a'):
        label = clean_text(link.get_text(' ', strip=True))
        if not label:
            continue

        if not EMAIL_RE.search(label) and not re.search(r'https?://|www\.', label, flags=re.IGNORECASE):
            candidate_link_names.append(label)
        text = text.replace(label, ' ')

    text = EMAIL_RE.sub(' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\b(E-?mail|Website)\s*:\s*', ' ', text, flags=re.IGNORECASE)
    cleaned = clean_text(text).strip(' -:;')

    # Some RBM entity lists put the entity name itself in an internal link.
    return cleaned or (candidate_link_names[0] if candidate_link_names else '')


def append_record(reg, list_name, name, category='', website='', email='', license_type=''):
    if not clean_text(name):
        return

    sqldict['Name'].append(clean_text(name))
    sqldict['CoType'].append(clean_text(category))
    sqldict['License_Type'].append(clean_text(license_type))
    sqldict['Website'].append(clean_text(website))
    sqldict['Email'].append(clean_text(email))
    sqldict['Cntry'].append('MW')
    sqldict['ListProcessDate'].append(processdate)
    sqldict['RegCtry'].append('MW')
    sqldict['RegCode'].append('RBM')
    sqldict['ListCode'].append(reg.split(' ')[-1])
    sqldict['RegulationType'].append('Regulated')
    sqldict['ListName'].append(list_name)

    bourange_same_length_array(sqldict)

In [5]:
#------------------------------------------------ Begin_Scraping Functions ----------------------------------------
def scrape_banks_and_other_financial_institutions(soup, reg, list_name):
    current_category = 'Banks'
    before_first_entity = True

    for node in iter_section_nodes(
        soup,
        'List of Banks & Other Financial Institutions',
        ['Annual Reports', "What's New"],
    ):
        category = category_from_node(node)
        if category:
            if category not in (list_name, 'Information on Banks and other financial institutions'):
                current_category = category
            continue

        if node.name != 'li':
            continue

        name = clean_entity_name(node)
        if not name:
            continue

        if normalise_heading(name) in ('banks', 'credit reference bureaux'):
            current_category = name
            continue

        # The RBM page lists banks first, followed by credit reference bureaux.
        if before_first_entity and not current_category:
            current_category = 'Banks'
        before_first_entity = False

        website, email = extract_contact_links(node)
        append_record(reg, list_name, name, category=current_category, website=website, email=email)


def scrape_pensions_and_insurance(soup, reg, list_name):
    current_category = ''

    for node in iter_section_nodes(
        soup,
        'Register of Licensed Entities',
        ["What's New"],
    ):
        category = category_from_node(node)
        if category and normalise_heading(category) != normalise_heading('Register of Licensed Entities'):
            current_category = category
            continue

        if node.name == 'li':
            name = clean_entity_name(node)
            if not name:
                continue

            website, email = extract_contact_links(node)
            append_record(reg, list_name, name, category=current_category, website=website, email=email)
            continue

        if node.name == 'table':
            rows = node.find_all('tr')
            for row in rows:
                cells = [clean_text(cell.get_text(' ', strip=True)) for cell in row.find_all(['td', 'th'])]
                if len(cells) < 1 or normalise_heading(cells[0]) == 'name':
                    continue

                name = cells[0]
                fund_type = cells[1] if len(cells) > 1 else ''
                append_record(
                    reg,
                    list_name,
                    name,
                    category=current_category or 'Pension Funds',
                    license_type=fund_type,
                )

In [6]:
#------------------------------------------------ Begin_Main ----------------------------------------
for reg, url in regdict.items():
    list_name = Typology[reg]
    print(f"[INFO] : Working {reg} - {list_name}")

    soup = fetch_soup(url)
    list_code = reg.split(' ')[-1]

    if list_code == '1':
        scrape_banks_and_other_financial_institutions(soup, reg, list_name)
    elif list_code == '2':
        scrape_pensions_and_insurance(soup, reg, list_name)
    else:
        print(f"[WARN] : No scraper configured for {reg}")

    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))

    print(f"[INFO] : Completed {reg}")
    sleep(1)

[INFO] : Working MW RBM 1 - List of Banks & Other Financial Institutions
[INFO] : Completed MW RBM 1
[INFO] : Working MW RBM 2 - Pensions & Insurance Supervision
[INFO] : Completed MW RBM 2


In [7]:
#------------------------------------------------ Save DataFrame to Excel ----------------------------------------
os.chdir(scriptfolder)
df = pd.DataFrame(sqldict)
df.to_excel(filename, 'SQL Ready', index=False)
sleep(3)
print(f"[INFO] : Excel file '{filename}' saved successfully")

C:\Users\wuj1\AppData\Local\Temp\2\ipykernel_50336\1997204381.py:4: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(filename, 'SQL Ready', index=False)


[INFO] : Excel file 'MW RBM SQL Ready 2026-05-05 16.54.51.xlsx' saved successfully


In [8]:
#------------------------------------------------ Data Verification ----------------------------------------
print('=' * 80)
print('DATA INTEGRITY & CONSISTENCY VERIFICATION')
print('=' * 80)
print(f"Total rows collected: {len(df)}")
print(f"Total columns: {len(df.columns)}")

if not df.empty:
    print('\nRows by list:')
    print(df.groupby(['ListCode', 'ListName']).size().rename('Count'))

    print('\nRows by category:')
    print(df.groupby(['ListCode', 'CoType']).size().rename('Count'))

    print('\nRegCtry values:', sorted(df['RegCtry'].dropna().unique()))
    print('RegCode values:', sorted(df['RegCode'].dropna().unique()))
    print('\nSample rows:')
    display(df[['Name', 'CoType', 'License_Type', 'Website', 'Email', 'ListCode', 'ListName']].head(10))
else:
    print('[WARN] : No rows collected')

DATA INTEGRITY & CONSISTENCY VERIFICATION
Total rows collected: 156
Total columns: 44

Rows by list:
ListCode  ListName                                    
1         List of Banks & Other Financial Institutions     11
2         Pensions & Insurance Supervision                145
Name: Count, dtype: int64

Rows by category:
ListCode  CoType                                   
1         Banks                                         9
          Credit Reference Bureaux                      2
2         BANCASSURANCE AGENTS / Agents for Brokers     5
          General Insurance Agents                     60
          General Insurance Companies                   8
          Insurance Brokers                            27
          Life Insurance Companies                      6
          Pension Funds                                31
          Pension Services Companies                    7
          Reinsurance Companies                         1
Name: Count, dtype: int64

RegCtry values: 

,Name,CoType,License_Type,Website,Email,ListCode,ListName
0,CDH INVESTMENT BANK,Banks,,http://www.cdh-malawi.com/,,1,List of Banks & Other Financial Institutions
1,ECOBANK,Banks,,http://www.ecobank.com/,,1,List of Banks & Other Financial Institutions
2,FDH BANK,Banks,,http://www.fdh.co.mw/,,1,List of Banks & Other Financial Institutions
3,FIRST CAPITAL BANK,Banks,,,,1,List of Banks & Other Financial Institutions
4,NATIONAL BANK OF MALAWI,Banks,,http://www.natbank.co.mw,,1,List of Banks & Other Financial Institutions
5,NBS BANK,Banks,,http://www.nbsmw.com/,,1,List of Banks & Other Financial Institutions
6,NEDBANK,Banks,,http://www.nedbank.co.mw/,,1,List of Banks & Other Financial Institutions
7,STANDARD BANK MALAWI,Banks,,http://www.standardbank.co.mw,,1,List of Banks & Other Financial Institutions
8,NEW FINANCE BANK MALAWI,Banks,,http://www.nfb.mw,,1,List of Banks & Other Financial Institutions
9,Credit Data CRB Limited,Credit Reference Bureaux,,,,1,List of Banks & Other Financial Institutions
